Connecting to an external cloud storage.

In [0]:
# mounting - Outdated
# this is outdated and replaced by the volume concept
dbutils.fs.mount(
  source = "s3a://my-bucket", #s3 bucket link
  mount_point = "/mnt/my-bucket", #dbfs path in databricks
  extra_configs = {"fs.s3a.access.key":"<ACCESS_KEY>", "fs.s3a.secret.key":"<SECRET_KEY>"} #credentials for s3 bucket
)


In [0]:
%sql
-- credential
CREATE STORAGE CREDENTIAL my_storage_cred
WITH AZURE_MANAGED_IDENTITY 'my-managed-identity'; --my-managed-identity will have JSON/XML file with credentials

-- base location -similar to mounting
CREATE EXTERNAL LOCATION my_ext_loc
URL 'abfss://my-container@my-storage-account.dfs.core.windows.net/'
WITH CREDENTIAL my_storage_cred;

-- exact folder location as volume
CREATE VOLUME my_catalog.my_schema.my_volume
LOCATION 'abfss://my-container@my-storage-account.dfs.core.windows.net/my-folder/';


In [0]:
#**********************************************************************
#Ingesting files and generating report in delta table
#**********************************************************************

CATALOG   = "employeeproject"
SCHEMA    = "default"
VOLUME    = f"/Volumes/{CATALOG}/{SCHEMA}/samplefiles"

EMPLOYEES_CSV   = f"{VOLUME}/employees.csv"
TRANSACTIONS_JSON = f"{VOLUME}/sales_transactions.json"
DEPT_TARGETS_PARQUET = f"{VOLUME}/department_targets.parquet"

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.revenue_report"

print(f"Source Volume  : {VOLUME}")
print(f"Target Table   : {TARGET_TABLE}")



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType, DoubleType
)

employees_schema = StructType([
    StructField("employee_id",  IntegerType(), False),
    StructField("first_name",   StringType(),  True),
    StructField("last_name",    StringType(),  True),
    StructField("department",   StringType(),  True),
    StructField("hire_date",    StringType(),  True),   # cast below
    StructField("base_salary",  DoubleType(),  True),
    StructField("region",       StringType(),  True),
])

df_employees = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "false")
         .schema(employees_schema)
         .csv(EMPLOYEES_CSV)
         .withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))
         .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
)

print(f"employees rows : {df_employees.count()}")
df_employees.printSchema()
df_employees.display()

In [0]:
df_raw_json = spark.read.option("multiLine", "true").json(TRANSACTIONS_JSON)
df_raw_json.printSchema()

df_transactions = (
    df_raw_json
    .withColumn("deal", F.explode("deals"))
    .select(
        F.col("transaction_id"),
        F.col("employee_id"),
        F.col("quarter"),
        F.col("revenue").cast(DoubleType()).alias("total_revenue"),
        F.col("deal.deal_id").alias("deal_id"),
        F.col("deal.client").alias("client_name"),
        F.col("deal.value").cast(DoubleType()).alias("deal_value"),
    )
)

print(f"transactions rows (after explode): {df_transactions.count()}")
df_transactions.display()

In [0]:
df_targets = spark.read.parquet(DEPT_TARGETS_PARQUET)

print(f"targets rows : {df_targets.count()}")
df_targets.printSchema()
df_targets.display()

In [0]:

# 4a. Aggregate revenue per employee per quarter
df_emp_revenue = (
    df_transactions
    .groupBy("employee_id", "quarter")
    .agg(
        F.sum("total_revenue").alias("quarterly_revenue"),
        F.count("deal_id").alias("deal_count"),
        F.collect_list("client_name").alias("clients"),
    )
)

# 4b. Enrich with employee master
df_enriched = (
    df_emp_revenue
    .join(df_employees, on="employee_id", how="left")
)

# 4c. Bring in department-level targets & bonus %
df_with_targets = (
    df_enriched
    .join(df_targets.select("department", "annual_target", "bonus_pct", "cost_center"),
          on="department", how="left")
)

# 4d. Compute bonus and attainment
df_report = (
    df_with_targets
    .withColumn(
        "attainment_pct",
        F.round((F.col("quarterly_revenue") / (F.col("annual_target") / 4)) * 100, 2)
    )
    .withColumn(
        "bonus_earned",
        F.round(F.col("base_salary") * F.col("bonus_pct") * (F.col("attainment_pct") / 100), 2)
    )
    .withColumn("report_ts", F.current_timestamp())
    .select(
        "employee_id",
        "full_name",
        "department",
        "region",
        "cost_center",
        "quarter",
        "quarterly_revenue",
        "deal_count",
        "clients",
        "annual_target",
        "attainment_pct",
        "base_salary",
        "bonus_pct",
        "bonus_earned",
        "hire_date",
        "report_ts",
    )
    .orderBy("quarter", "department", "full_name")
)

print(f"Final report rows : {df_report.count()}")
df_report.display()

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

(
    df_report
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}").display()
spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY quarter, department").display()

spark.sql(f"""
    SELECT
        department,
        quarter,
        COUNT(*)              AS headcount,
        ROUND(SUM(quarterly_revenue), 0)  AS total_revenue,
        ROUND(AVG(attainment_pct), 1)     AS avg_attainment_pct,
        ROUND(SUM(bonus_earned), 0)       AS total_bonus_pool
    FROM {TARGET_TABLE}
    GROUP BY department, quarter
    ORDER BY quarter, department
""").display()